In [ ]:
import os
import sys

os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable


In [2]:
from pyspark.sql import SparkSession
spark = (
    SparkSession.builder.appName("OlistDataInspection").master("local[*]").getOrCreate()
)

In [4]:
spark.sparkContext.setLogLevel("Error")

In [3]:
print("Spark version:", spark.version)

spark.range(5).show()

Spark version: 3.5.9
+---+
| id|
+---+
|  0|
|  1|
|  2|
|  3|
|  4|
+---+



In [5]:
orders = spark.read.csv("../data/olist_orders_dataset.csv",header=True,inferSchema=True)

In [6]:
orders.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)



In [7]:
orders.show(5,truncate=False)

+--------------------------------+--------------------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+
|order_id                        |customer_id                     |order_status|order_purchase_timestamp|order_approved_at  |order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|
+--------------------------------+--------------------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+
|e481f51cbdc54678b7cc49136f2d6af7|9ef432eb6251297304e76186b10a928d|delivered   |2017-10-02 10:56:33     |2017-10-02 11:07:15|2017-10-04 19:55:00         |2017-10-10 21:25:13          |2017-10-18 00:00:00          |
|53cdb2fc8bc7dce0b6741e2150273451|b0830fb4747a6c6d20dea0b8c802d7ef|delivered   |2018-07-24 20:41:37     |2018-07-26 03:24:27|2018-07-26 14:3

In [10]:
orders.count()


99441

In [11]:
orders.columns

['order_id',
 'customer_id',
 'order_status',
 'order_purchase_timestamp',
 'order_approved_at',
 'order_delivered_carrier_date',
 'order_delivered_customer_date',
 'order_estimated_delivery_date']

Transformations:Tell Spark what to do


In [13]:
orders.select("order_id","customer_id","order_status").show(5,truncate=False)

+--------------------------------+--------------------------------+------------+
|order_id                        |customer_id                     |order_status|
+--------------------------------+--------------------------------+------------+
|e481f51cbdc54678b7cc49136f2d6af7|9ef432eb6251297304e76186b10a928d|delivered   |
|53cdb2fc8bc7dce0b6741e2150273451|b0830fb4747a6c6d20dea0b8c802d7ef|delivered   |
|47770eb9100c2d0c44946d9cf07ec65d|41ce2a54c0b03bf3443c3d931a367089|delivered   |
|949d5b44dbf5de918fe9c16f97b45f8a|f88197465ea7920adcdbec7375364d82|delivered   |
|ad21c59c0840e6cb83a9ceb5573f8159|8ab97904e6daea8866dbdbc4fb7aad2c|delivered   |
+--------------------------------+--------------------------------+------------+
only showing top 5 rows



In [14]:
delivered_orders = orders.filter(orders.order_status == "delivered")
delivered_orders.show(5,truncate=False)

+--------------------------------+--------------------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+
|order_id                        |customer_id                     |order_status|order_purchase_timestamp|order_approved_at  |order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|
+--------------------------------+--------------------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+
|e481f51cbdc54678b7cc49136f2d6af7|9ef432eb6251297304e76186b10a928d|delivered   |2017-10-02 10:56:33     |2017-10-02 11:07:15|2017-10-04 19:55:00         |2017-10-10 21:25:13          |2017-10-18 00:00:00          |
|53cdb2fc8bc7dce0b6741e2150273451|b0830fb4747a6c6d20dea0b8c802d7ef|delivered   |2018-07-24 20:41:37     |2018-07-26 03:24:27|2018-07-26 14:3

In [15]:
delivered_orders.count()

96478

Actions:Ask Spark to produce a result

In [16]:
print("Total orders:",orders.count())
print("Delivered orders:",delivered_orders.count())

Total orders: 99441
Delivered orders: 96478


In [19]:
cancelled_orders = orders.filter(orders.order_status=="canceled")
cancelled_orders.show(5,truncate=False)


+--------------------------------+--------------------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+
|order_id                        |customer_id                     |order_status|order_purchase_timestamp|order_approved_at  |order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|
+--------------------------------+--------------------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+
|1b9ecfe83cdc259250e1a8aca174f0ad|6d6b50b66d79f80827b6d96751528d30|canceled    |2018-08-04 14:29:27     |2018-08-07 04:10:26|NULL                        |NULL                         |2018-08-14 00:00:00          |
|714fb133a6730ab81fa1d3c1b2007291|e3fe72696c4713d64d3c10afe71e75ed|canceled    |2018-01-26 21:34:08     |2018-01-26 21:58:39|2018-01-29 22:3

In [20]:
print("Cancelled orders:", cancelled_orders.count())
print("Order statuses:")
orders.groupBy("order_status").count().show()

Cancelled orders: 625
Order statuses:
+------------+-----+
|order_status|count|
+------------+-----+
|     shipped| 1107|
|    canceled|  625|
|    invoiced|  314|
|     created|    5|
|   delivered|96478|
| unavailable|  609|
|  processing|  301|
|    approved|    2|
+------------+-----+

